# Aula 16 — Vanishing e exploding gradients

Laboratório reproduzível em **NumPy puro** para acompanhar produtos de Jacobianos, ativações e gradientes por camada. Não há autograd, downloads ou dados externos.

**Pergunta experimental:** como pequenas contrações ou expansões locais se acumulam com a profundidade, e quais métricas distinguem o fenômeno de um gradiente apenas pequeno?

> Execute as células em ordem. O arquivo publicado permanece sem outputs; uma cópia é executada durante a validação.

## Dependências e contrato

- Python ≥ 3.11
- NumPy ≥ 1.26
- Matplotlib ≥ 3.8
- nbformat ≥ 5.9 para validação do arquivo

Seed canônica: `20260916`. Os dados são sintéticos, os geradores são explícitos e todas as operações usam `float64`, exceto a contraprova declarada de underflow em `float32`.

In [ ]:
import platform
import warnings

import matplotlib
import matplotlib.pyplot as plt
import numpy as np

warnings.filterwarnings("error")
SEED = 20260916

print("Python:", platform.python_version())
print("NumPy:", np.__version__)
print("Matplotlib:", matplotlib.__version__)
print("Seed:", SEED)
assert tuple(map(int, np.__version__.split(".")[:2])) >= (1, 26)

## 1. Cadeia escalar: o efeito exponencial

Para $h_L=c^Lh_0$, o gradiente $partial h_L/partial h_0=c^L$. Fatores moderados tornam-se extremos quando repetidos.

In [ ]:
depth_scalar = 50
factors = np.array([0.8, 1.0, 1.2], dtype=np.float64)
scalar_gradients = factors**depth_scalar

for factor, gradient in zip(factors, scalar_gradients):
    print(f"c={factor:.1f}: c^{depth_scalar}={gradient:.12e}, log10={np.log10(gradient):.6f}")

assert np.isclose(scalar_gradients[0], 1.4272476927059638e-5)
assert scalar_gradients[1] == 1.0
assert np.isclose(scalar_gradients[2], 9100.438150002134)
assert scalar_gradients[2] / scalar_gradients[0] > 1e8

## 2. Underflow não é a origem do vanishing

A contração matemática existe antes de o computador arredondar o resultado para zero. Repetimos $0{,}5$ em `float32` e `float64`; o underflow é esperado e tratado explicitamente.

In [ ]:
def repeated_product(factor, depth, dtype):
    value = np.array(1.0, dtype=dtype)
    with np.errstate(under="ignore"):
        for _ in range(depth):
            value *= np.array(factor, dtype=dtype)
    return value


product32 = repeated_product(0.5, 200, np.float32)
product64 = repeated_product(0.5, 200, np.float64)
print("0,5^200 em float32:", product32)
print("0,5^200 em float64:", f"{float(product64):.12e}")
assert product32 == 0.0
assert product64 > 0.0
assert np.isclose(product64, 0.5**200)
assert 0.5**50 > 0.0  # vanishing pode ser representável e ainda operacionalmente minúsculo


## 3. Produto matricial com espectro controlado

Uma matriz ortogonal $Q$ preserva a norma. Para $J=sQ$, todos os valores singulares são $s$ e um produto de $L$ cópias altera a norma exatamente por $s^L$, salvo arredondamento.

In [ ]:
rng = np.random.default_rng(SEED)
raw = rng.normal(size=(16, 16))
Q, R = np.linalg.qr(raw)
Q = Q @ np.diag(np.sign(np.diag(R)))
assert np.allclose(Q.T @ Q, np.eye(16), atol=1e-12)

direction = rng.normal(size=16)
direction /= np.linalg.norm(direction)
matrix_depth = 30
matrix_ratios = {}
for scale in (0.8, 1.0, 1.2):
    vector = direction.copy()
    for _ in range(matrix_depth):
        vector = (scale * Q.T) @ vector
    matrix_ratios[scale] = np.linalg.norm(vector)
    expected = scale**matrix_depth
    print(f"s={scale:.1f}: razão={matrix_ratios[scale]:.12e}, esperada={expected:.12e}")
    assert np.isclose(matrix_ratios[scale], expected, rtol=1e-12, atol=1e-14)

singular_values = np.linalg.svd(1.2 * Q, compute_uv=False)
assert np.allclose(singular_values, 1.2, atol=1e-12)

## 4. Um Jacobiano completo, sem autograd

Para duas camadas `tanh`, montamos $J_1=D_1W_1$ e $J_2=D_2W_2$. O VJP obtido pelo produto explícito deve coincidir com o backward camada a camada.

In [ ]:
rng_j = np.random.default_rng(SEED + 1)
W1 = rng_j.normal(0.0, 0.4, size=(3, 3))
W2 = rng_j.normal(0.0, 0.4, size=(3, 3))
x = rng_j.normal(size=3)
upstream = rng_j.normal(size=3)

z1 = W1 @ x
a1 = np.tanh(z1)
z2 = W2 @ a1
a2 = np.tanh(z2)
D1 = np.diag(1.0 - a1**2)
D2 = np.diag(1.0 - a2**2)
J1 = D1 @ W1
J2 = D2 @ W2

explicit = (J2 @ J1).T @ upstream
g2 = upstream * (1.0 - a2**2)
g1 = W2.T @ g2
manual = W1.T @ (g1 * (1.0 - a1**2))
jacobian_error = np.max(np.abs(explicit - manual))

print("VJP explícito:", explicit)
print("Backward manual:", manual)
print(f"Erro máximo: {jacobian_error:.3e}")
assert jacobian_error < 1e-15

## 5. Instrumentação de uma MLP profunda

Na convenção com amostras em linhas, o forward usa `A @ W`. O backward aplica primeiro a derivada da ativação e depois `@ W.T`. Medimos RMS para comparar camadas do mesmo experimento sem depender do tamanho do lote.

In [ ]:
def rms(array):
    array = np.asarray(array, dtype=np.float64)
    return float(np.sqrt(np.mean(array**2)))


def activate(z, name):
    if name == "tanh":
        return np.tanh(z)
    if name == "relu":
        return np.maximum(z, 0.0)
    if name == "linear":
        return z
    raise ValueError(name)


def derivative(z, name):
    if name == "tanh":
        value = np.tanh(z)
        return 1.0 - value**2
    if name == "relu":
        return (z > 0.0).astype(np.float64)
    if name == "linear":
        return np.ones_like(z)
    raise ValueError(name)


def weight_matrix(rng, fan_in, fan_out, scheme, scale=1.0):
    if scheme == "small":
        std = 0.02
    elif scheme == "large":
        std = 1.0
    elif scheme == "xavier":
        std = np.sqrt(2.0 / (fan_in + fan_out))
    elif scheme == "he":
        std = np.sqrt(2.0 / fan_in)
    else:
        raise ValueError(scheme)
    return rng.normal(0.0, scale * std, size=(fan_in, fan_out))


def instrumented_mlp(X, depth, width, activation, scheme, seed, scale=1.0):
    rng_local = np.random.default_rng(seed)
    activations = [X.copy()]
    weights, preactivations, activation_stats = [], [], []
    A = X.copy()
    for _ in range(depth):
        W = weight_matrix(rng_local, A.shape[1], width, scheme, scale)
        Z = A @ W
        A = activate(Z, activation)
        weights.append(W)
        preactivations.append(Z)
        activations.append(A)
        activation_stats.append({
            "rms": rms(A),
            "second_moment": float(np.mean(A**2)),
            "zero_fraction": float(np.mean(A == 0.0)),
            "saturation": float(np.mean(np.abs(A) > 0.99)),
        })

    G = rng_local.normal(size=A.shape)
    G /= rms(G)
    gradient_rms = [None] * (depth + 1)
    gradient_rms[-1] = rms(G)
    derivative_rms = [None] * depth
    for index in range(depth - 1, -1, -1):
        local = derivative(preactivations[index], activation)
        derivative_rms[index] = rms(local)
        G = (G * local) @ weights[index].T
        gradient_rms[index] = rms(G)

    return {
        "weights": weights,
        "preactivations": preactivations,
        "activations": activations,
        "activation_stats": activation_stats,
        "gradient_rms": np.asarray(gradient_rms),
        "derivative_rms": np.asarray(derivative_rms),
    }


assert np.isclose(rms(np.array([3.0, 3.0, 3.0, 3.0])), 3.0)
assert activate(np.array([-1.0, 2.0]), "relu").tolist() == [0.0, 2.0]

## 6. `tanh`: saturação e contração

Comparamos pesos pequenos, Xavier e pesos grandes em 30 camadas. Pesos grandes podem elevar a escala afim e, simultaneamente, saturar `tanh`; por isso “peso grande” não implica automaticamente gradiente grande.

In [ ]:
batch, width, depth = 1024, 96, 30
X = np.random.default_rng(SEED + 2).normal(size=(batch, width))
tanh_runs = {}
for scheme in ("small", "xavier", "large"):
    run = instrumented_mlp(X, depth, width, "tanh", scheme, SEED + 3)
    tanh_runs[scheme] = run
    ratio = run["gradient_rms"][0] / run["gradient_rms"][-1]
    saturation = run["activation_stats"][0]["saturation"]
    print(
        f"{scheme:7s}: RMS g_entrada={run['gradient_rms'][0]:.6e}, "
        f"razão={ratio:.6e}, saturação camada 1={saturation:.3%}"
    )

assert tanh_runs["small"]["gradient_rms"][0] < 1e-15
assert tanh_runs["large"]["activation_stats"][0]["saturation"] > 0.7
assert tanh_runs["xavier"]["activation_stats"][0]["saturation"] < 0.02
assert all(np.isfinite(run["gradient_rms"]).all() for run in tanh_runs.values())

### Gráfico — gradiente `tanh` por profundidade

O eixo vertical logarítmico revela contrações que uma escala linear esconderia.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.6))
positions = np.arange(depth + 1)
for scheme, run in tanh_runs.items():
    ax.plot(positions, run["gradient_rms"], marker="o", markersize=2.5, label=scheme)
ax.set_yscale("log")
ax.set_xlabel("Posição (0 = entrada; 30 = saída)")
ax.set_ylabel("RMS do gradiente")
ax.set_title("Fluxo do gradiente em MLPs tanh não treinadas")
ax.grid(True, which="both", alpha=0.25)
ax.legend()
plt.tight_layout()
plt.show()
print("Texto alternativo: três curvas logarítmicas mostram forte contração com pesos pequenos, melhor fluxo com Xavier e efeito combinado de pesos grandes e saturação.")

## 7. ReLU com He subescalado, correto e superescalado

Mantemos dados, arquitetura, ativação e seed. Alteramos somente um multiplicador da escala He: `0.5`, `1.0` ou `1.5`.

In [ ]:
relu_runs = {}
for scale in (0.5, 1.0, 1.5):
    run = instrumented_mlp(X, depth, width, "relu", "he", SEED + 4, scale=scale)
    relu_runs[scale] = run
    g_ratio = run["gradient_rms"][0] / run["gradient_rms"][-1]
    a_last = run["activation_stats"][-1]["second_moment"]
    print(f"escala={scale:.1f}: razão gradiente={g_ratio:.6e}, E[A30²]={a_last:.6e}")

sub_ratio = relu_runs[0.5]["gradient_rms"][0]
he_ratio = relu_runs[1.0]["gradient_rms"][0]
super_ratio = relu_runs[1.5]["gradient_rms"][0]
assert sub_ratio < he_ratio / 1e6
assert super_ratio > he_ratio * 1e3
assert relu_runs[0.5]["activation_stats"][-1]["second_moment"] < 1e-12
assert relu_runs[1.5]["activation_stats"][-1]["second_moment"] > 1e6
assert all(np.isfinite(run["gradient_rms"]).all() for run in relu_runs.values())

### Gráfico — contração e expansão em ReLU

Os três casos permanecem finitos, mas dois já são operacionalmente patológicos. Isso demonstra por que `isfinite` é necessário e insuficiente.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.4))
layers = np.arange(1, depth + 1)
for scale, run in relu_runs.items():
    moments = [s["second_moment"] for s in run["activation_stats"]]
    axes[0].plot(layers, moments, marker="o", markersize=2.5, label=f"{scale:.1f}× He")
    axes[1].plot(positions, run["gradient_rms"], marker="o", markersize=2.5, label=f"{scale:.1f}× He")
for axis in axes:
    axis.set_yscale("log")
    axis.grid(True, which="both", alpha=0.25)
    axis.legend()
axes[0].set(xlabel="Camada", ylabel="E[A²]", title="Forward")
axes[1].set(xlabel="Posição do gradiente", ylabel="RMS(G)", title="Backward")
plt.tight_layout()
plt.show()
print("Texto alternativo: dois painéis logarítmicos mostram 0,5× He colapsando, He próximo de escala útil e 1,5× He crescendo muitas ordens no forward e backward.")

## 8. Norma L2 e RMS não são intercambiáveis

Duplicar um tensor com os mesmos valores preserva RMS, mas aumenta a norma L2 por $sqrt{2}$. Por isso shapes diferentes exigem contexto.

In [ ]:
base_gradient = np.array([1.0, -2.0, 3.0, -4.0])
duplicated_gradient = np.concatenate([base_gradient, base_gradient])
l2_base = np.linalg.norm(base_gradient)
l2_duplicated = np.linalg.norm(duplicated_gradient)
rms_base = rms(base_gradient)
rms_duplicated = rms(duplicated_gradient)

print(f"L2 base={l2_base:.6f}; duplicado={l2_duplicated:.6f}; razão={l2_duplicated/l2_base:.6f}")
print(f"RMS base={rms_base:.6f}; duplicado={rms_duplicated:.6f}")
assert np.isclose(l2_duplicated / l2_base, np.sqrt(2.0))
assert rms_base == rms_duplicated

## 9. Clipping global por norma

O clipping só reduz vetores acima do limite. Verificamos que a norma explosiva é limitada, a direção é preservada e um gradiente pequeno permanece inalterado.

In [ ]:
def clip_global_norm(arrays, max_norm, eps=1e-12):
    if max_norm <= 0:
        raise ValueError("max_norm deve ser positivo")
    total = np.sqrt(sum(float(np.sum(np.asarray(g, dtype=np.float64)**2)) for g in arrays))
    factor = min(1.0, max_norm / (total + eps))
    return [np.asarray(g, dtype=np.float64) * factor for g in arrays], total, factor


explosive = [np.array([30.0, 40.0])]
clipped, norm_before, clip_factor = clip_global_norm(explosive, 5.0)
norm_after = np.linalg.norm(clipped[0])
cosine = float(explosive[0] @ clipped[0] / (np.linalg.norm(explosive[0]) * norm_after))

vanishing = [np.array([0.003, 0.004])]
small_clipped, small_before, small_factor = clip_global_norm(vanishing, 5.0)

print(f"Explosivo: norma {norm_before:.6f} -> {norm_after:.6f}; fator={clip_factor:.6f}; cosseno={cosine:.12f}")
print(f"Pequeno: norma {small_before:.6f}; fator={small_factor:.6f}")
assert np.isclose(norm_before, 50.0)
assert np.isclose(norm_after, 5.0, atol=1e-12)
assert np.isclose(clip_factor, 0.1, atol=1e-12)
assert np.isclose(cosine, 1.0)
assert small_factor == 1.0
assert np.array_equal(small_clipped[0], vanishing[0])

## 10. Repetição em cinco seeds

He é uma regra de escala, não uma promessa de identidade entre execuções. Resumimos o logaritmo da razão do gradiente de entrada para separar tendência e variabilidade.

In [ ]:
seed_ratios = []
for offset in range(5):
    run = instrumented_mlp(X, depth, width, "relu", "he", SEED + 100 + offset)
    seed_ratios.append(run["gradient_rms"][0] / run["gradient_rms"][-1])
seed_ratios = np.asarray(seed_ratios)
log_ratios = np.log10(seed_ratios)

print("Razões por seed:", np.array2string(seed_ratios, precision=6))
print(f"log10 — média={log_ratios.mean():.6f}, desvio={log_ratios.std(ddof=1):.6f}")
assert seed_ratios.size == 5
assert np.all(np.isfinite(seed_ratios))
assert np.all(seed_ratios > 0.0)
assert log_ratios.max() - log_ratios.min() < 4.0

## 11. Auditoria final

Os contratos cobrem fórmulas conhecidas, produto de Jacobianos, underflow declarado, instrumentação, saturação, contração, expansão, métricas comparáveis, clipping e robustez a seeds.

In [ ]:
audit = {
    "cadeia contrai": np.isclose(scalar_gradients[0], 0.8**50),
    "cadeia neutra": scalar_gradients[1] == 1.0,
    "cadeia expande": np.isclose(scalar_gradients[2], 1.2**50),
    "underflow float32 detectado": product32 == 0.0,
    "float64 preserva produto": product64 > 0.0,
    "matriz ortogonal": np.allclose(Q.T @ Q, np.eye(16), atol=1e-12),
    "espectro controlado": np.allclose(singular_values, 1.2, atol=1e-12),
    "VJP manual correto": jacobian_error < 1e-15,
    "tanh pequena desaparece": tanh_runs["small"]["gradient_rms"][0] < 1e-15,
    "tanh grande satura": tanh_runs["large"]["activation_stats"][0]["saturation"] > 0.7,
    "Xavier reduz saturação inicial": tanh_runs["xavier"]["activation_stats"][0]["saturation"] < 0.02,
    "ReLU subescalada desaparece": sub_ratio < he_ratio / 1e6,
    "ReLU superescalada explode": super_ratio > he_ratio * 1e3,
    "casos ainda finitos": all(np.isfinite(run["gradient_rms"]).all() for run in relu_runs.values()),
    "RMS independe da duplicação": rms_base == rms_duplicated,
    "L2 depende do tamanho": np.isclose(l2_duplicated / l2_base, np.sqrt(2.0)),
    "clipping limita norma": np.isclose(norm_after, 5.0, atol=1e-12),
    "clipping preserva direção": np.isclose(cosine, 1.0),
    "clipping não aumenta pequeno": small_factor == 1.0,
    "cinco seeds válidas": seed_ratios.size == 5 and np.all(np.isfinite(seed_ratios)),
}

for name, passed in audit.items():
    assert passed, name

print(f"Auditoria: {sum(audit.values())}/{len(audit)} contratos aprovados.")

## Conclusões

- Produtos escalares e matriciais confirmaram contração e expansão exponenciais.
- O underflow em `float32` ocorreu depois que o gradiente já era matematicamente minúsculo.
- O VJP por Jacobiano total coincidiu com o backward manual camada a camada.
- Pesos pequenos apagaram o gradiente `tanh`; pesos grandes saturaram a ativação.
- Em ReLU, `0.5× He` colapsou e `1.5× He` explodiu, embora ambos permanecessem finitos.
- L2 mudou com o tamanho do tensor; RMS permaneceu comparável.
- Clipping limitou a explosão e preservou direção, mas não aumentou o gradiente pequeno.
- Cinco seeds mostraram variabilidade sem invalidar a tendência.

Na Aula 17, o backward diagnosticado será aplicado a mini-batches, epochs e embaralhamento reprodutível.